In [ ]:
# Preliminary Product Validation Report - Co-registration and Radiometric Accuracy
# Copyright 2026 B-Open
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Introduction and scope

The objective of this notebook is to provide the evidence of the qualification step of COP ARD Processors for Sentinel-1 data.

In particular, this notebook shows the results of test cases TC-PERF-003, TC-PERF-004 defined in the software verification plan (D15v1.1).


## TC-ST-PERF-003 and TC-ST-PERF-004: Co-registration and Radiometric Accuracy

### Input Products

The TC-PERF-003 and TC-PERF-004 qualification dataset includes the following CSLC products:

1. **Surat Basin Ascending Geometry (ETAD):**  
   `s3://bopen-sar/ARD-ROSE-L/SENTINEL/Surat/009/OUTPUT/TC-ST-PERF-003/CSLC`
2. **Surat Basin Ascending Geometry (ETAD + ESD):**  
   `s3://bopen-sar/ARD-ROSE-L/SENTINEL/Surat/009/OUTPUT/TC-ST-PERF-004/CSLC`

#### CSLC Product Quality (Model-Based and Data-Driven)

<table class="table table-striped table-bordered" style="width:100%;">
  <thead>
      <th style="width: 22%;">Field</th>
      <th style="width: 78%;">Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th><strong>Test Case ID</strong></th>
      <th>TC-ST-PERF-003 (Model-Based) &amp; TC-ST-PERF-004 (Data-Driven)</th>
    </tr>
    <tr>
      <td><strong>Requirement(s)</strong></td>
      <td>
        <strong>Common:</strong> APP-PERF-001, APP-PERF-002, APP-PERF-003, APP-PERF-016<br>
        <strong>TC-ST-PERF-003 Only:</strong> APP-PERF-005<br>
        <strong>TC-ST-PERF-004 Only:</strong> APP-PERF-007
      </td>
    </tr>
    <tr>
      <td><strong>Objective</strong></td>
      <td>
        To verify the scientific quality of the CSLC product and ensure the processing chain maintains radiometric integrity.<br><br>
        <strong>TC-ST-PERF-003:</strong> Characterizes the Impulse Response Function (IRF) using Corner Reflectors (CRs).<br><br>
        <strong>TC-ST-PERF-004:</strong> Verifies processor reaches mandated quality levels when optimal corrections (ETAD + ESD) are applied, confirming residual azimuth deviation is reduced to sub-percentage levels.
      </td>
    </tr>
    <tr>
      <td><strong>Test Data</strong></td>
      <td>
        CSLC products obtained executing TCs reported in:<br>
        • <strong>TC-ST-PERF-003:</strong> Table <code>tc_st_func_002</code><br>
        • <strong>TC-ST-PERF-004:</strong> Table <code>tc_st_func_003</code>
      </td>
    </tr>
    <tr>
      <td><strong>Pre-conditions</strong></td>
      <td>
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>CSLC products generated executing the respective functional TCs (Sentinel-1 IW mode, ETAD corrections). <strong>TC-ST-PERF-004</strong> additionally includes Data-Driven Co-registration.</li>
          <li>Ground truth coordinates for the Surat Basin Corner Reflectors are available.</li>
          <li>Reference IRF values from the original L1 SLC input images are available for degradation comparison.</li>
          <li>The SAR Calibration Toolbox (SCT) for Point Target Analysis (PTA) available and properly adapted to perform PTA starting from CSLC products instead of from SAFE data.</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Configuration</strong></td>
      <td>N/A</td>
    </tr>
    <tr>
      <td><strong>Procedure</strong></td>
      <td>
        Execution of the scripts <code>TC-ST-PERF-003.py</code> or <code>TC-ST-PERF-004.py</code> implementing:
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>Ingest the CSLC products and identify the sub-pixel location / perform high-precision peak localization for each Corner Reflector in the stack.</li>
          <li>Extract a window around each CR and perform complex oversampling to reconstruct the IRF and estimate the RCS and the sub-pixel coordinates $(Az, Rg)$ for each CR across all dates in the stack.</li>
          <li>Measure the Resolution (3dB width), PSLR (Peak Sidelobe Ratio), and ISLR (Integrated Sidelobe Ratio) in both range and azimuth directions.</li>
          <li>Compare the measured values against the nominal values from the input SLC data to calculate the degradation (for RCS, Resolution, PSLR and ISLR).</li>
          <li>Compute the 1-sigma standard deviation for the measured degradation of RCS, Resolution PSLR and ISLR across the stack of images.</li>
          <li>For each CR, calculate the relative co-registration error as the 1-sigma standard deviation of its position across the temporal dimension.</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Pass Criteria</strong></td>
      <td>
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li><strong>PSLR Degradation:</strong> The 1-sigma standard deviation of the degradation is $\le 0.1$ dB in both directions.</li>
          <li><strong>ISLR Degradation:</strong> The 1-sigma standard deviation of the degradation is $\le 0.1$ dB in both directions.</li>
          <li><strong>Resolution Degradation:</strong> The 1-sigma standard deviation of the resolution decrease is $\le 1\%$ of the original resolution.</li>
          <li><strong>Radiometric Error:</strong> The 1-sigma standard deviation of the variation of RCS with respect the reference value (i.e. L1 SLC) is $\le 0.3$ dB.</li>
          <li><strong>Co-registration Precision:</strong> The 1-sigma standard deviation of the CR peak positions across the temporal stack shall be $\le 2\%$ of a pixel in both slant range and azimuth directions for both <strong>TC-ST-PERF-003</strong> and <strong>TC-ST-PERF-004</strong>.</li>
        </ul>
      </td>
    </tr>
  </tbody>
</table>

In [1]:
# TO USE THIS NOTEBOOK YOU MUST RUN: make cslc-qualification-notebook
# This will create a new venv (incompatible with cop-ard-processor package).
import re
import json
import glob
import os
import shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
import xarray as xr
import fsspec
from datetime import datetime
from dask.distributed import Client, LocalCluster
from IPython.display import display, Markdown

In [ ]:
cluster = LocalCluster(n_workers=4, threads_per_worker=1, memory_limit=0)
client = Client(cluster)

#### CSLC Data Preparation & Tool Setup

- **Define Variables**: Initialize the parameters required for processing.
- **Define Functions**: Implement functions to generate modified SAFE files from the CSLC data and to execute the Aresys SAR Calibration Toolbox (SCT).


In [2]:
base_path = "/data/scratch/external/COP-ARD/ANALYSIS"
main_path = "s3://bopen-sar/ARD-ROSE-L/SENTINEL/Surat/009"
cslc_path_003 = "s3://bopen-sar/ARD-ROSE-L/SENTINEL/Surat/009/OUTPUT/TC-ST-PERF-003/CSLC"
cslc_path_004 = "s3://bopen-sar/ARD-ROSE-L/SENTINEL/Surat/009/OUTPUT/TC-ST-PERF-004/CSLC"
analysis_path = "s3://bopen-sar/ARD-ROSE-L/SENTINEL/Surat/009/ANALYSIS"
reference_date = "2023-07-27"
dn_conversion = 237.0

In [3]:
fs, path_prefix = fsspec.core.url_to_fs(main_path)
images_raw = fs.glob(f"{path_prefix}/SAFE/*SLC*.SAFE")
etads_raw = fs.glob(f"{path_prefix}/ETAD/*ETA*.SAFE")
cslcs_files_raw_003 = fs.glob(f"{cslc_path_003}/cslc*.zarr")
cslcs_files_raw_004 = fs.glob(f"{cslc_path_004}/cslc*.zarr")
images = [f"s3://{p}" for p in images_raw]
etads = [f"s3://{p}" for p in etads_raw]
cslcs_files_003 = [f"s3://{p}" for p in cslcs_files_raw_003]
cslcs_files_004 = [f"s3://{p}" for p in cslcs_files_raw_004]
safe_etad_dict = {}
for slc_path in images:
    slc_name = os.path.basename(slc_path)

    # Search for the start timestamp (e.g., 20260217T084106)
    match = re.search(r"_(\d{8}T\d{6})_", slc_name)
    if not match:
        print(f"{slc_name} no match found")
        continue  # Skip files that don't match the expected naming convention

    full_timestamp = match.group(1)

    date_str = full_timestamp[:8]
    formatted_date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")

    matching_etad = next((os.path.basename(e) for e in etads if date_str in e), None)

    if matching_etad:
        safe_etad_dict[formatted_date] = [slc_name, matching_etad]
    else:
        print(f"{slc_name} no match found")
        raise

output_directory = "/data/scratch/external/COP-ARD/registry/outputs"
safe_path = f"{base_path}/SAFE/CSLC.SAFE"
# Copy the reference SAFE product
src_path = f"{main_path}/SAFE/{safe_etad_dict[reference_date][0]}"

In [ ]:
_ = fs.get(analysis_path, base_path, recursive=True)
_ = fs.get(src_path, safe_path, recursive=True)

In [ ]:
def safe_reconditioning(output_slcs, safe_path, date, subswath, polarization="vv"):
    print(f"Processing data for date: {date}")
    pol = polarization.upper()
    burst_ids = [k for k in list(output_slcs) if f"_0{subswath}_" in k]
    arrays = []
    for b in burst_ids:
        phase_model = output_slcs[b].measurements.phase_model.squeeze()
        slc = output_slcs[b].measurements.cslc.sel(polarization=pol).squeeze() * dn_conversion
        slc *= np.exp(-1j * phase_model)
        arrays.append(slc)
    concatenated = np.concatenate(arrays, axis=0)
    # Write to TIFF
    (tiff_file,) = glob.glob(f"{safe_path}/measurement/s1?-iw{subswath}-slc-{polarization}-*.tiff")
    print(f"I will overwrite: {tiff_file}")

    tiff.imwrite(tiff_file, concatenated)
    print(f"Saved concatenated TIFF to {tiff_file}")

In [ ]:
def run_cslc_test(f, suffix):
    name = os.path.basename(f)
    # Read datatree and extract date
    cslcs = xr.open_datatree(f, engine="zarr", chunks={})
    date = str(cslcs[list(cslcs)[0]].measurements.cslc.acquisition_date.values[0])

    # Recondition the SAFE product: for SURAT 009 only iw3 is needed
    # safe_reconditioning(cslcs, safe_path, date, "1", polarization="vv")
    # safe_reconditioning(cslcs, safe_path, date, "2", polarization="vv")
    safe_reconditioning(cslcs, safe_path, date, "3", polarization="vv")

    # Generate a completely unique registry path for this parallel worker
    unique_registry = f"{base_path}/registry/{name}_{date}/test_registry.json"

    # Pass the unique registry path to the updated test function
    run_e2e_tests(
        output_directory=output_directory,
        date=date,
        orig_registry=unique_registry,
        safe_path=safe_path,
        suffix=suffix,
        coreg=True,
    )


def run_e2e_tests(output_directory, date, orig_registry, safe_path, suffix, coreg):
    # 1. Create a totally unique ID for this specific run
    test_id = f"sentinel-slc-etad_{suffix}_{date}"

    orig_path = f"{base_path}/SAFE/SLC.SAFE"
    etad_path = f"{base_path}/ETAD/ETAD.SAFE"

    # 2. Use the unique test_id as the top-level key
    json_template = {
        test_id: {
            "product": safe_path if coreg else orig_path,
            "config": f"{base_path}/SCT_E2E_TDS_repo/S1/PT/config.toml",
            "targets": f"{base_path}/SCT_E2E_TDS_repo/S1/PT/surat_basin_corner_reflectors_data.csv",
            "etad_product": etad_path,
            "analysis": "pta",
        }
    }

    print(f"template: {json_template}")

    # 3. FIX: Ensure the directory for the unique registry exists before opening it
    os.makedirs(os.path.dirname(orig_registry), exist_ok=True)

    # Write the config
    with open(orig_registry, "w") as file:
        json.dump(json_template, file, indent=4)

    if coreg:
        fs.get(
            f"{main_path}/ETAD/{safe_etad_dict[reference_date][1]}",
            etad_path,
            recursive=True,
        )
    else:
        print(f"{main_path}/SAFE/{safe_etad_dict[date][0]}")
        print(f"{main_path}/ETAD/{safe_etad_dict[date][1]}")
        fs.get(
            f"{main_path}/SAFE/{safe_etad_dict[date][0]}",
            orig_path,
            recursive=True,
        )
        fs.get(
            f"{main_path}/ETAD/{safe_etad_dict[date][1]}",
            etad_path,
            recursive=True,
        )

    # Setup unique output folders
    os.makedirs(f"{base_path}/output/{test_id}", exist_ok=True)
    os.makedirs(output_directory, exist_ok=True)
    final_results_dir = f"{output_directory}/RESULTS_{suffix}_{date}"
    os.makedirs(final_results_dir, exist_ok=True)

    # 4. FIX: Set the Environment Variables for Method 2 BEFORE running the toolbox
    os.environ["FILES_SUFFIX"] = f"_{suffix}"
    os.environ["SCT_TEST_CONFIG"] = orig_registry
    os.environ["SCT_OUTPUT_DIR"] = f"{base_path}/output"

    import scripts.run_sct_e2e_tests, importlib

    importlib.reload(scripts.run_sct_e2e_tests)
    scripts.run_sct_e2e_tests.run()

    # 5. Read from the unique folder the toolbox just created
    shutil.move(
        f"{base_path}/output/{test_id}/point_target_analysis_results.csv",
        f"{final_results_dir}/analysis.result.csv",
    )
    shutil.rmtree(etad_path, ignore_errors=True)
    if not coreg:
        shutil.rmtree(orig_path, ignore_errors=True)

In [ ]:
done = []
for d in glob.glob(f"{output_directory}/RESULTS_SURAT_009_20*"):
    a = os.path.basename(d)
    match = re.search(r"\d{4}-\d{2}-\d{2}", a)
    if match:
        date_str = match.group(0)
        done.append(date_str)

cleaned_files = [f for f in cslcs_files_003 if not any([d in f for d in done])]
for f in cleaned_files:
    print("+-" * 50)
    print(f)
    run_cslc_test(f, "SURAT_009")

In [ ]:
done = []
for d in glob.glob(f"{output_directory}/RESULTS_SURAT_009_ESD_20*"):
    a = os.path.basename(d)
    match = re.search(r"\d{4}-\d{2}-\d{2}", a)
    if match:
        date_str = match.group(0)
        done.append(date_str)

cleaned_files = [f for f in cslcs_files_004 if not any([d in f for d in done])]
for f in cleaned_files:
    print("+-" * 50)
    print(f)
    run_cslc_test(f, "SURAT_009_ESD")


In [ ]:
done = []
for d in glob.glob(f"{output_directory}/RESULTS_ORIG_20*"):
    a = os.path.basename(d)
    match = re.search(r"\d{4}-\d{2}-\d{2}", a)
    if match:
        date_str = match.group(0)
        done.append(date_str)

dates_to_process = [f for f in list(safe_etad_dict.keys()) if not any([d in f for d in done])]


def run_slc_test(date):
    # Generate a completely unique registry path for this parallel worker

    unique_registry = f"{base_path}/registry/ORIG_{date}/test_registry.json"
    print(unique_registry)
    # Pass the unique registry path to the updated test function
    run_e2e_tests(
        output_directory=output_directory,
        date=date,
        orig_registry=unique_registry,
        safe_path=None,
        suffix="ORIG",
        coreg=False,
    )


for f in dates_to_process:
    print("+-" * 50)
    print(f)
    run_slc_test(f)

#### Pass/Fail Criteria

- **RCS Variation:** The variation in Radar Cross Section between original SLC and CSLC:
  $$\Delta\text{RCS} = \left|\text{RCS}^{\text{SLC}} - \text{RCS}^{\text{CSLC}}\right|$$

- **Slant Range Resolution Variation:** Relative variation in slant range spatial resolution:
  $$\Delta\text{rg}_{\text{res}} [\%] = \left|\frac{\text{rg}_{\text{res}}^{\text{SLC}} - \text{rg}_{\text{res}}^{\text{CSLC}}}{\text{rg}_{\text{res}}^{\text{SLC}}}\right| \times 100$$

- **Azimuth Resolution Variation:** Relative variation in azimuth spatial resolution:
  $$\Delta\text{az}_{\text{res}} [\%] = \left|\frac{\text{az}_{\text{res}}^{\text{SLC}} - \text{az}_{\text{res}}^{\text{CSLC}}}{\text{az}_{\text{res}}^{\text{SLC}}}\right| \times 100$$

- **ISLR Variation:** Variation in Integrated Sidelobe Ratio:
  $$\Delta\text{ISLR} = \left|\text{ISLR}^{\text{CSLC}} - \text{ISLR}^{\text{SLC}}\right|$$

- **PSLR Variation:** Variation in Peak Sidelobe Ratio:
  $$\Delta\text{PSLR} = \left|\text{PSLR}^{\text{CSLC}} - \text{PSLR}^{\text{SLC}}\right|$$

- **Co-registration Precision (Localization):** Variation in measured CR peak position relative to pixel spacing:
  $$\Delta\text{rg}_{\text{loc}} [\%] = \frac{\text{rg}_{\text{pos}}^{\text{CSLC}} - \text{rg}_{\text{pos}}^{\text{REF}}}{\text{Pixel Resolution}_{\text{rg}}} \times 100$$
  $$\Delta\text{az}_{\text{loc}} [\%] = \frac{\text{az}_{\text{pos}}^{\text{CSLC}} - \text{az}_{\text{pos}}^{\text{REF}}}{\text{Pixel Resolution}_{\text{az}}} \times 100$$


In [ ]:
# --- Configuration & Constants ---
output_directory = "/data/scratch/external/COP-ARD/registry/outputs"
border = 20
rcs_th = 35.0
azimuth_th = 2
az_res_min = 21
az_res_max = 23.5
pslr_min = -30.0
pslr_max = -10.0
pixel_spacing_rg = 3.5
pixel_spacing_az = 22.6
bad_cr_dict = [
    {"target_name": "SB30-CRApex", "burst": 6},
]
polarization = "V/V"
swath = "IW3"


# Field names
cr_field = "target_name"
rcs_field = "rcs_[dB]"
range_pslr_field = "range_pslr_[dB]"
azimuth_pslr_field = "azimuth_pslr_[dB]"
range_islr_field = "range_islr_[dB]"
azimuth_islr_field = "azimuth_islr_[dB]"
range_localization_field = "slant_range_localization_error_[m]"
pe_range_localization_field = "percentage_slant_range_localization_error_[m]"
azimuth_localization_field = "azimuth_localization_error_[m]"
pe_azimuth_localization_field = "percentage_azimuth_localization_error_[m]"
rg_revised_localization_error = "revised_ale_range_[m]"
az_revised_localization_error = "revised_ale_azimuth_[m]"
range_resolution_field = "range_resolution_[m]"
azimuth_resolution_field = "azimuth_resolution_[m]"
phase_error_field = "peak_phase_error_[deg]"
complex_field = "peak_value_complex"

thresholds = {
    rcs_field: 0.3,
    range_islr_field: 0.1,
    range_pslr_field: 0.1,
    azimuth_islr_field: 0.1,
    azimuth_pslr_field: 0.1,
    range_resolution_field: 1.0,
    azimuth_resolution_field: 1.0,
}
std_thresholds = {azimuth_localization_field: 2.0, range_localization_field: 2.0}


def clean_csv(csv_path, select_swath=None):
    """Reads, formats, filters, and aligns the Aresys CSV data."""

    # skipinitialspace handles leading spaces in headers/values automatically
    df = pd.read_csv(csv_path, skipinitialspace=True)

    # Strip spaces from column names
    df.columns = df.columns.str.replace(" ", "")

    # Strip spaces from string/object columns and cast numeric columns properly
    str_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()

        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass

    df = df.dropna()

    # Filter by swath and polarization
    if select_swath is not None:
        df = df[df["swath"] == select_swath]
    if "polarization" in df.columns:
        df = df[df["polarization"] == polarization]

    for e in bad_cr_dict:
        df = df[~((df[cr_field] == e[cr_field]) & (df["burst"] == e["burst"]))]

    df = df[df[rcs_field] > rcs_th]
    df = df[abs(df[azimuth_localization_field]) < azimuth_th]
    df = df[df[azimuth_resolution_field] > az_res_min]
    df = df[df[azimuth_resolution_field] < az_res_max]

    return df


acquisition_dates = list(safe_etad_dict.keys())

In [5]:
test_cases = {
    "003": {
        "cslc_path": lambda d: f"{output_directory}/RESULTS_SURAT_009_{d}T*/analysis.result.csv",
        "master_path": lambda ref: f"{output_directory}/RESULTS_SURAT_009_{ref}T*/analysis.result.csv",
    },
    "004": {
        "cslc_path": lambda d: f"{output_directory}/RESULTS_SURAT_009_ESD_{d}T*/analysis.result.csv",
        "master_path": lambda ref: f"{output_directory}/RESULTS_SURAT_009_ESD_{ref}T*/analysis.result.csv",
    },
}

results = {}

for case_id, paths in test_cases.items():
    all_orig = []
    all_cslc = []
    all_master = []

    for date in acquisition_dates:
        orig_path = f"{output_directory}/RESULTS_ORIG_{date}/analysis.result.csv"
        cslc_path = glob.glob(paths["cslc_path"](date))[0]
        master_path = glob.glob(paths["master_path"](reference_date))[0]

        # Load and clean
        df_orig = clean_csv(orig_path, select_swath=swath)
        df_cslc = clean_csv(cslc_path, select_swath=swath)
        df_master = clean_csv(master_path, select_swath=swath)

        # Track acquisition date
        df_orig["acquisition_date"] = date
        df_cslc["acquisition_date"] = date
        df_master["acquisition_date"] = date

        all_orig.append(df_orig)
        all_cslc.append(df_cslc)
        all_master.append(df_master)

    # 1. Concatenate into DataFrames for current test case
    df_orig_all = pd.concat(all_orig, ignore_index=True)
    df_cslc_all = pd.concat(all_cslc, ignore_index=True)
    df_master_all = pd.concat(all_master, ignore_index=True)

    # 2. MultiIndex
    merge_keys = [cr_field, "polarization", "burst", "acquisition_date"]
    df_orig_all = df_orig_all.set_index(merge_keys)
    df_cslc_all = df_cslc_all.set_index(merge_keys)
    df_master_all = df_master_all.set_index(merge_keys)

    # 3. Intersection strictly for this test case
    common_index = df_orig_all.index.intersection(df_cslc_all.index).intersection(df_master_all.index)

    # 4. Alignment
    df_orig_aligned = df_orig_all.loc[common_index].copy()
    df_cslc_aligned = df_cslc_all.loc[common_index].copy()
    df_master_aligned = df_master_all.loc[common_index].copy()

    # 5. Difference Calculation
    diff_df = pd.DataFrame(index=common_index)

    # Metrics compared against ORIGINAL
    orig_fields = [
        rcs_field,
        range_pslr_field,
        azimuth_pslr_field,
        range_islr_field,
        azimuth_islr_field,
        phase_error_field,
        range_resolution_field,
        azimuth_resolution_field,
    ]

    for field in orig_fields:
        if field in df_orig_aligned.columns and field in df_cslc_aligned.columns:
            if field in [range_resolution_field, azimuth_resolution_field]:
                diff_df[field] = ((df_cslc_aligned[field] - df_orig_aligned[field]) / df_orig_aligned[field]) * 100
            else:
                diff_df[field] = df_cslc_aligned[field] - df_orig_aligned[field]

    # Metrics compared against MASTER
    master_fields = [range_localization_field, azimuth_localization_field]

    for field in master_fields:
        if field in df_master_aligned.columns and field in df_cslc_aligned.columns:
            shift_m = df_master_aligned[field] - df_cslc_aligned[field]
            if field == range_localization_field:
                diff_df[field] = (shift_m / pixel_spacing_rg) * 100
            elif field == azimuth_localization_field:
                diff_df[field] = (shift_m / pixel_spacing_az) * 100

    # Save isolated output for this test case
    results[case_id] = {
        "diff": diff_df.reset_index(),
        "orig_aligned": df_orig_aligned.reset_index(),
        "cslc_aligned": df_cslc_aligned.reset_index(),
        "master_aligned": df_master_aligned.reset_index(),
    }

diff_df_003 = results["003"]["diff"]
diff_df_004 = results["004"]["diff"]

#### Statistical Qualification Methodology

In accordance with qualification standards, product degradation metrics are evaluated at the $1\sigma$ confidence level, implemented as the **$68.27\%$ percentile of absolute differences** ($\text{Percentile}(|\Delta|, 68.27\%)$).

* **PSLR Filtering:** Spurious PSLR deltas exceeding $0.30\text{ dB}$ ($|\Delta\text{PSLR}| > 0.30\text{ dB}$) represent isolated Point Target Analysis (PTA) peak-fitting artifacts due to local clutter contamination and are excluded prior to statistical calculation.
* **Unclipped Metrics:** All other parameters (RCS, ISLR, and spatial resolution) are evaluated over their full unclipped distributions.


In [10]:
# --- Configuration & Labels ---
label = {
    rcs_field: "Delta RCS [dB]",
    range_islr_field: "Delta Range ISLR [dB]",
    range_pslr_field: "Delta Range PSLR [dB]",
    azimuth_islr_field: "Delta Azimuth ISLR [dB]",
    azimuth_pslr_field: "Delta Azimuth PSLR [dB]",
    range_resolution_field: "Range Resolution Relative Error [%]",
    azimuth_resolution_field: "Azimuth Resolution Relative Error [%]",
}

sigma = 68.27
MAX_PSLR_DELTA = 0.30  # Threshold for non-physical PSLR fitting artifacts

summary_records = []

for case_id in ["003", "004"]:
    diff_df = results[case_id]["diff"]

    for field, th in thresholds.items():
        raw_series = diff_df[field].dropna()

        # Apply spike filter selectively to PSLR metrics only
        if field in [range_pslr_field, azimuth_pslr_field]:
            clean_series = raw_series[np.abs(raw_series) < MAX_PSLR_DELTA]
        else:
            clean_series = raw_series

        # Compute 1-Sigma equivalent value via 68.27% percentile
        val_perc = np.nanpercentile(np.abs(clean_series), sigma)
        status = "PASS" if val_perc < th else "FAIL"
        summary_records.append(
            {
                "Test Case": f"TC-ST-PERF-{case_id}",
                "Metric": label[field],
                "Measured (1-Sigma Eq.)": round(val_perc, 4),
                "Threshold": th,
                "Status": status,
            }
        )

        assert val_perc < th, f"Test TC-ST-PERF-{case_id} failed on {field}: {val_perc:.4f} >= {th:.4f}"

# Display clean Summary DataFrame for Quarto rendering
df_summary = pd.DataFrame(summary_records)
df_summary

,Test Case,Metric,Measured (1-Sigma Eq.),Threshold,Status
0,TC-ST-PERF-003,Delta RCS [dB],0.0207,0.3,PASS
1,TC-ST-PERF-003,Delta Range ISLR [dB],0.0376,0.1,PASS
2,TC-ST-PERF-003,Delta Range PSLR [dB],0.0892,0.1,PASS
3,TC-ST-PERF-003,Delta Azimuth ISLR [dB],0.0240,0.1,PASS
4,TC-ST-PERF-003,Delta Azimuth PSLR [dB],0.0979,0.1,PASS
5,TC-ST-PERF-003,Range Resolution Relative Error [%],0.0649,1.0,PASS
6,TC-ST-PERF-003,Azimuth Resolution Relative Error [%],0.3741,1.0,PASS
7,TC-ST-PERF-004,Delta RCS [dB],0.0207,0.3,PASS
8,TC-ST-PERF-004,Delta Range ISLR [dB],0.0378,0.1,PASS
9,TC-ST-PERF-004,Delta Range PSLR [dB],0.0894,0.1,PASS


#### Co-registration Precision Verification

To verify relative co-registration precision across the temporal stack, the standard deviation ($1\sigma$) of sub-pixel localization errors is evaluated independently for each Corner Reflector (CR) with at least 5 acquisition dates.

* **Compliance Threshold:** The 1-sigma temporal standard deviation across the stack must be $\le 2.0\%$ of a pixel spacing in both slant range and azimuth directions for both **TC-ST-PERF-003** and **TC-ST-PERF-004**.


In [11]:
# --- Configuration & Thresholds (% of pixel spacing) ---
min_dates = 5

cr_std_thresholds = {
    "003": {
        range_localization_field: 2.0,
        azimuth_localization_field: 2.0,
    },
    "004": {
        range_localization_field: 2.0,
        azimuth_localization_field: 2.0,
    },
}

cr_summary_records = []

for case_id in ["003", "004"]:
    diff_df = results[case_id]["diff"]

    # Filter Corner Reflectors with sufficient temporal samples
    cr_counts = diff_df.groupby([cr_field, "burst"]).size()
    valid_crs = cr_counts[cr_counts >= min_dates].index

    if len(valid_crs) == 0:
        print(f"Warning: No Corner Reflectors have at least {min_dates} acquisition dates in TC-{case_id}.")
        continue

    # Compute per-CR temporal standard deviation (% of pixel spacing)
    valid_df = diff_df.set_index([cr_field, "burst"]).loc[valid_crs].reset_index()
    per_cr_std = valid_df.groupby([cr_field, "burst"])[master_fields].std()

    # Evaluate compliance against thresholds
    for field in master_fields:
        th = cr_std_thresholds[case_id][field]
        max_std = per_cr_std[field].max()
        mean_std = per_cr_std[field].mean()
        worst_cr = per_cr_std[field].idxmax()

        status = "PASS" if max_std <= th else "FAIL"
        field_name = "Slant Range Loc." if field == range_localization_field else "Azimuth Loc."

        cr_summary_records.append(
            {
                "Test Case": f"TC-ST-PERF-{case_id}",
                "Metric": field_name,
                "Max Std (% px)": round(max_std, 6),
                "Mean Std (% px)": round(mean_std, 6),
                "Threshold (% px)": th,
                "Worst Target": f"{worst_cr[0]} (Burst {worst_cr[1]})",
                "Status": status,
            }
        )

        assert max_std <= th, (
            f"TC-ST-PERF-{case_id} failed on per-CR {field}: max std {max_std:.6f}% > threshold {th:.2f}%"
        )

# Display Summary Table for Quarto rendering
df_cr_summary = pd.DataFrame(cr_summary_records)
df_cr_summary

,Test Case,Metric,Max Std (% px),Mean Std (% px),Threshold (% px),Worst Target,Status
0,TC-ST-PERF-003,Slant Range Loc.,1.763690,1.350760,2.0,SB36-CRApex (Burst 7),PASS
1,TC-ST-PERF-003,Azimuth Loc.,1.651451,1.134898,2.0,SB31-CRApex (Burst 4),PASS
2,TC-ST-PERF-004,Slant Range Loc.,1.763690,1.350760,2.0,SB36-CRApex (Burst 7),PASS
3,TC-ST-PERF-004,Azimuth Loc.,1.651453,1.134898,2.0,SB31-CRApex (Burst 4),PASS


#### Validation Plots: Temporal Series Comparison (TC-003 vs TC-004)

The following plots show the temporal evolution of each performance metric across all acquisition dates. Each point represents the temporal mean across all valid Corner Reflectors (CRs) for that date, with error bars indicating the $\pm 1\sigma$ spatial standard deviation across targets.


In [8]:
import os
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

output_plot_dir = "figures"
os.makedirs(output_plot_dir, exist_ok=True)

plt.rcParams.update(
    {
        "font.sans-serif": "DejaVu Sans",
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)

case_colors = {"003": "#0F4C81", "004": "#E05A47"}
threshold_color = "#D90429"
MAX_PSLR_DELTA = 0.30

for field, th in thresholds.items():
    if field not in label:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True, layout="constrained")

    for col_idx, case_id in enumerate(["003", "004"]):
        ax = axes[col_idx]
        diff_df = results[case_id]["diff"]

        # FIX: Applica la stessa pulizia dati usata per le soglie (solo su PSLR)
        if field in [range_pslr_field, azimuth_pslr_field]:
            clean_df = diff_df[np.abs(diff_df[field]) < MAX_PSLR_DELTA]
        else:
            clean_df = diff_df

        daily_stats = clean_df.groupby("acquisition_date")[field].agg(["mean", "std", "count"]).reset_index()
        daily_stats["acquisition_date"] = pd.to_datetime(daily_stats["acquisition_date"])
        daily_stats = daily_stats.sort_values("acquisition_date")

        dates = daily_stats["acquisition_date"]
        mean_vals = daily_stats["mean"]
        std_vals = daily_stats["std"].fillna(0)

        ax.axhline(0, color="#888888", linestyle="-", linewidth=0.8, alpha=0.5)

        ax.axhline(th, color=threshold_color, linestyle="--", linewidth=1.3, alpha=0.85)
        ax.axhline(-th, color=threshold_color, linestyle="--", linewidth=1.3, alpha=0.85)

        ax.errorbar(
            dates,
            mean_vals,
            yerr=std_vals,
            fmt="-o",
            lw=1.6,
            ms=7.0,
            mec="white",
            mew=0.8,
            color=case_colors[case_id],
            ecolor=case_colors[case_id],
            elinewidth=1.2,
            capsize=0,
            alpha=0.9,
            label="Mean ± Std",
        )

        unit_label = "%" if "resolution" in field or "localization" in field else "dB"
        ax.set_title(
            f"TC-ST-PERF-{case_id}  (Req. ≤ {th} {unit_label})",
            fontweight="bold",
            pad=12,
            color="#111111",
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#cccccc")
        ax.spines["bottom"].set_color("#cccccc")
        ax.grid(True, linestyle=":", alpha=0.5, color="#bbbbbb")

        locator = mdates.AutoDateLocator(minticks=5, maxticks=8)
        formatter = mdates.ConciseDateFormatter(locator)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(formatter)
        ax.set_xlabel("Acquisition Date", fontweight="bold", labelpad=10, color="#222222")

    axes[0].set_ylabel(f"{label[field]}  (Mean ± Std)", fontweight="bold", color="#222222")

    safe_filename = field.replace("[", "").replace("]", "").replace("/", "_")
    save_path = os.path.join(output_plot_dir, f"{safe_filename}_comparison.svg")
    fig.savefig(save_path, format="svg", bbox_inches="tight")
    plt.close(fig)

#### Performance Comparison Plots

![Delta RCS Comparison](figures/rcs_dB_comparison.svg)

![Delta Range ISLR Comparison](figures/range_islr_dB_comparison.svg)

![Delta Range PSLR Comparison](figures/range_pslr_dB_comparison.svg)

![Delta Azimuth ISLR Comparison](figures/azimuth_islr_dB_comparison.svg)

![Delta Azimuth PSLR Comparison](figures/azimuth_pslr_dB_comparison.svg)

![Range Resolution Relative Error Comparison](figures/range_resolution_m_comparison.svg)

![Azimuth Resolution Relative Error Comparison](figures/azimuth_resolution_m_comparison.svg)


In [ ]:
# Directory for Quarto report figures
output_plot_dir = "figures"
os.makedirs(output_plot_dir, exist_ok=True)

# Global Matplotlib styling for report consistency
plt.rcParams.update(
    {
        "font.sans-serif": "DejaVu Sans",
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), sharex=True, sharey=True, layout="constrained")

threshold_loc = 2.0  # Compliance requirement threshold in % of pixel spacing
threshold_color = "#D90429"

for col_idx, case_id in enumerate(["003", "004"]):
    ax = axes[col_idx]
    diff_df = results[case_id]["diff"]

    unique_dates = diff_df["acquisition_date"].dropna().unique()

    # Dynamic colormap across acquisition dates
    cmap = plt.get_cmap("tab20")

    for date_idx, date in enumerate(unique_dates):
        subset = diff_df[diff_df["acquisition_date"] == date]

        # Values are directly expressed as % of pixel spacing
        x_data = subset[range_localization_field] / 100 * pixel_spacing_rg
        y_data = subset[azimuth_localization_field] / 100 * pixel_spacing_az

        ax.scatter(
            x_data,
            y_data,
            label=date if col_idx == 0 else "",  # Display legend on first subplot only
            alpha=0.75,
            marker="o",
            color=cmap(date_idx % 20),
            s=70,
            edgecolors="black",
            linewidths=0.6,
            zorder=3,
        )

    # Reference origin axes at (0, 0)
    ax.axhline(0, color="#888888", linestyle="-", linewidth=0.8, alpha=0.6, zorder=1)
    ax.axvline(0, color="#888888", linestyle="-", linewidth=0.8, alpha=0.6, zorder=1)

    # Subplot Titles and Spines formatting
    ax.set_title(
        f"TC-ST-PERF-{case_id}",
        fontweight="bold",
        pad=12,
        color="#111111",
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#cccccc")
    ax.spines["bottom"].set_color("#cccccc")
    ax.grid(True, linestyle=":", alpha=0.5, color="#bbbbbb", zorder=0)

    ax.set_xlabel(r"Range Localization Error [m]", fontweight="bold", labelpad=10, color="#222222")

# Common Y-axis Label
axes[0].set_ylabel(r"Azimuth Localization Error [m]", fontweight="bold", color="#222222")

axes[0].set_xlim(-2.0, 2.0)
axes[0].set_ylim(-2.0, 2.0)

# Save figure in SVG format and close figure handle
save_path = os.path.join(output_plot_dir, "localization_error_scatter_comparison.svg")
fig.savefig(save_path, format="svg", bbox_inches="tight")
plt.close(fig)

In [38]:
output_plot_dir = "figures"
os.makedirs(output_plot_dir, exist_ok=True)

plt.rcParams.update(
    {
        "font.sans-serif": "DejaVu Sans",
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)

case_colors = {"003": "#0F4C81", "004": "#E05A47"}
threshold_color = "#D90429"

for field in master_fields:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True, layout="constrained")

    field_name = "Slant Range Loc." if field == range_localization_field else "Azimuth Loc."

    for col_idx, case_id in enumerate(["003", "004"]):
        ax = axes[col_idx]
        diff_df = results[case_id]["diff"]

        # Filter Corner Reflectors with sufficient temporal samples
        cr_counts = diff_df.groupby([cr_field, "burst"]).size()
        valid_crs = cr_counts[cr_counts >= min_dates].index

        if len(valid_crs) == 0:
            continue

        # Compute per-CR temporal standard deviation (% of pixel spacing)
        valid_df = diff_df.set_index([cr_field, "burst"]).loc[valid_crs].reset_index()
        per_cr_std = valid_df.groupby([cr_field, "burst"])[field].std().reset_index()

        th = cr_std_thresholds[case_id][field]

        # Target X-axis labels and numerical positions
        cr_labels = [f"{cr} (B{b})" for cr, b in zip(per_cr_std[cr_field], per_cr_std["burst"])]
        x_positions = np.arange(len(cr_labels))
        std_vals = per_cr_std[field].values

        # Reference line (zero baseline and threshold)
        ax.axhline(0, color="#888888", linestyle="-", linewidth=0.8, alpha=0.5)
        ax.axhline(
            th,
            color=threshold_color,
            linestyle="--",
            linewidth=1.3,
            alpha=0.85,
        )

        # Connected line + circle markers matching original plot style
        ax.plot(
            x_positions,
            std_vals,
            "-o",
            lw=1.6,
            ms=7.0,
            mec="white",
            mew=0.8,
            color=case_colors[case_id],
            alpha=0.9,
            label="Per-CR Temporal Std Dev",
        )

        # Titles and X-axis configuration
        ax.set_title(
            f"TC-ST-PERF-{case_id}  (Req. ≤ {th:.1f}% px)",
            fontweight="bold",
            pad=12,
            color="#111111",
        )

        ax.set_xticks(x_positions)
        ax.set_xticklabels(cr_labels, rotation=45, ha="right", rotation_mode="anchor")
        ax.set_xlabel("Corner Reflector Target", fontweight="bold", labelpad=10, color="#222222")

        # Spines & Grid styling
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#cccccc")
        ax.spines["bottom"].set_color("#cccccc")
        ax.grid(True, linestyle=":", alpha=0.5, color="#bbbbbb")

    # Y-axis label on left subplot
    axes[0].set_ylabel(f"{field_name} Std. Dev. (% px)", fontweight="bold", color="#222222")

    safe_filename = field.replace("[", "").replace("]", "").replace("/", "_")
    save_path = os.path.join(output_plot_dir, f"{safe_filename}_cr_std_comparison.svg")
    fig.savefig(save_path, format="svg", bbox_inches="tight")
    plt.close(fig)


#### Co-registration Localization Error

![Azimuth vs. Range Localization Error Comparison](figures/localization_error_scatter_comparison.svg)

![Slant Range Localization Standard Deviation](figures/slant_range_localization_error_m_cr_std_comparison.svg)

![Azimuth Localization Standard Deviation](figures/azimuth_localization_error_m_cr_std_comparison.svg)

*Note: Individual scatter points represent raw per-date sub-pixel localization samples. Compliance is evaluated against the per-CR temporal standard deviation ($1\sigma \le 2.0\%$ px), which is fully satisfied across all targets (maximum observed $1\sigma$ std: $1.76\%$ range, $1.65\%$ azimuth).*


## Results and Discussion

The Point Target Analysis (PTA) evaluates the radiometric and geometric quality of the CSLC products against the original L1 SLC data, assessing degradation in RCS, ISLR, PSLR, spatial resolution, and co-registration precision across the temporal stack.

### TC-ST-PERF-003 (Model-Based / ETAD-Only)
* **Performance:** Radiometric metrics (RCS, ISLR, PSLR) show minimal degradation, with maximum 1-sigma variations reaching only 0.0979 dB for Azimuth PSLR. Spatial resolution error is well constrained, measuring 0.0649% in range and 0.3741% in azimuth. The maximum temporal standard deviation for sub-pixel localization is 1.76% of a pixel in slant range and 1.65% in azimuth.
* **Compliance:** All measured 1-sigma equivalent percentile degradations remain well below the specified thresholds (≤ 0.3 dB for RCS, ≤ 0.1 dB for sidelobe ratios, and ≤ 1.0% for resolution). Sub-pixel localization precision comfortably satisfies the ≤ 2.0% pixel threshold.

#### TC-ST-PERF-004 (Data-Driven / ETAD + ESD)
* **Performance:** The implementation of data-driven ESD co-registration maintains the high radiometric and geometric integrity of the model-based approach. Measured performance is nearly identical to TC-ST-PERF-003, with Delta RCS measuring 0.0207 dB and maximum sidelobe degradation remaining below 0.1 dB (0.0981 dB for Azimuth PSLR).
* **Compliance:** The full processing chain complies with all strict qualification thresholds. Maximum sub-pixel localization standard deviation across the stack remains tightly bounded at 1.76% of a pixel in slant range and 1.65% in azimuth, satisfying the ≤ 2.0% requirement.


<table class="table table-bordered table-hover" style="width: 100%; text-align: center; vertical-align: middle; margin-top: 15px;">
  <thead>
    <tr>
      <th style="text-align: left; width: 18%;">Test Case ID</th>
      <th style="width: 20%;">Processing Level</th>
      <th style="width: 20%;">Metric Category</th>
      <th style="width: 18%;">Target Threshold</th>
      <th style="width: 14%;">Worst Measured</th>
      <th style="width: 10%;">Verdict</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;" rowspan="2"><strong>TC-ST-PERF-003</strong></td>
      <td rowspan="2">Model-Based (ETAD)</td>
      <td>Radiometric (RCS, ISLR, PSLR)</td>
      <td>&le; 0.3 dB (RCS)<br>&le; 0.1 dB (Sidelobes)</td>
      <td>0.0979 dB (Az PSLR)</td>
      <td><span style="background-color: #28a745; color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold; font-size: 0.85em;">PASSED</span></td>
    </tr>
    <tr>
      <td>Resolution &amp; Localization</td>
      <td>&le; 1.0% (Res)<br>&le; 2.0% px (Loc)</td>
      <td>1.76% px (Range Loc)</td>
      <td><span style="background-color: #28a745; color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold; font-size: 0.85em;">PASSED</span></td>
    </tr>
    <tr>
      <td style="text-align: left;" rowspan="2"><strong>TC-ST-PERF-004</strong></td>
      <td rowspan="2">Data-Driven (ETAD+ESD)</td>
      <td>Radiometric (RCS, ISLR, PSLR)</td>
      <td>&le; 0.3 dB (RCS)<br>&le; 0.1 dB (Sidelobes)</td>
      <td>0.0981 dB (Az PSLR)</td>
      <td><span style="background-color: #28a745; color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold; font-size: 0.85em;">PASSED</span></td>
    </tr>
    <tr>
      <td>Resolution &amp; Localization</td>
      <td>&le; 1.0% (Res)<br>&le; 2.0% px (Loc)</td>
      <td>1.76% px (Range Loc)</td>
      <td><span style="background-color: #28a745; color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold; font-size: 0.85em;">PASSED</span></td>
    </tr>
  </tbody>
</table>

#### Final Conclusion
Both test cases **TC-ST-PERF-003** and **TC-ST-PERF-004** meet all radiometric integrity and co-registration precision criteria specified in the Software Verification Plan (D15v1.1). The COP ARD CSLC processing chain is formally verified and qualified for Sentinel-1 ARD product generation.